# Evaluación del modelo final en el reporte Kaspersky (material complementario)

Equipo 55

Producto de difusión (Av6)

Script adicional al Avance 5, **independiente**: entrena desde cero, solo con el conjunto *train* de LADDER, el baseline del Avance 3, la regresión logística afinada del Avance 4 y el modelo final del Avance 5 (Stacking de regresión logística y Complement Naive Bayes calibrado), y los evalúa **una sola vez** en las 96 oraciones del reporte de Kaspersky, sin ajustar nada después.

**Declaración de uso de IA.** Este fragmento fue generado y adaptado con apoyo de Claude Sonnet 5.5 (Anthropic, 2026, https://claude.ai) para la evaluación del modelo final en el reporte Kaspersky. Fue ejecutado y revisado por el equipo; la responsabilidad final recae en los autores.

**Protocolo.** Los umbrales se fijan solo con train (predicciones fuera de pliegue, 5 pliegues, semilla 42): el umbral más alto que todavía da recall ≥ 0.95. No se usa test ni Kaspersky para elegir modelo ni umbral.

**Datos necesarios** (en la carpeta de trabajo de Colab, o subirlos cuando el notebook lo pida):
- `data.zip`, con `data/ladder/sentence_classification/{train,dev,test}.csv` (LADDER);
- `oraciones_kaspersky.csv` (columnas `id`, `tipo`, `texto`; las 96 oraciones tienen `tipo == "oracion"`);
- `anotacion_referencia.csv` (columnas `id`, `label_ref`, `ttps_gt`). Anotación de referencia preparada por el asesor con apoyo de IA: 55 oraciones de ataque y 41 de contexto.


In [1]:
# 1) Preparar los archivos (en Colab, si faltan se piden con el botón de carga)
import os, zipfile
from pathlib import Path

DATA_DIR = Path("data/ladder/sentence_classification")
CSV_KAS = ["oraciones_kaspersky.csv", "anotacion_referencia.csv"]

def listo():
    return DATA_DIR.joinpath("train.csv").exists() and all(os.path.exists(f) for f in CSV_KAS)

if not listo() and os.path.exists("data.zip"):
    zipfile.ZipFile("data.zip").extractall(".")

if not listo():
    try:
        from google.colab import files
        print("Sube data.zip, oraciones_kaspersky.csv y anotacion_referencia.csv")
        files.upload()
        if os.path.exists("data.zip"):
            zipfile.ZipFile("data.zip").extractall(".")
    except ImportError:
        pass

assert listo(), "Faltan archivos. Se necesitan data.zip (o la carpeta data/) y los dos CSV de Kaspersky."
print("Archivos listos.")

Archivos listos.


In [2]:
# 2) Datos de LADDER (solo train) y modelos, con los mismos hiperparámetros del Avance 4 y 5
import time, warnings
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, fbeta_score, f1_score, precision_recall_curve,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")

SEED = 42
META_RECALL = 0.95

tr = pd.read_csv(DATA_DIR / "train.csv", sep="\t")
ytr = tr.label.to_numpy()
print("train:", tr.shape)

# Baseline del Avance 3
baseline = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), sublinear_tf=True, min_df=1),
                         LogisticRegression(C=3, class_weight="balanced", solver="liblinear", max_iter=5000, random_state=SEED))

# LogReg afinado del Avance 4
lr_f = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 7), min_df=2, sublinear_tf=True),
                     LogisticRegression(C=3.382387201936218, penalty="l2", class_weight="balanced",
                                        solver="liblinear", max_iter=5000, random_state=SEED))

# Complement NB afinado del Avance 4, calibrado, y Stacking final del Avance 5
nb_f = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
                     ComplementNB(alpha=0.11040511903162252, norm=True))
nb_cal = CalibratedClassifierCV(nb_f, method="sigmoid", cv=5)
stacking = StackingClassifier(estimators=[("lr", lr_f), ("nb", nb_cal)], final_estimator=LogisticRegression(), cv=5)

train: (2214, 2)


In [3]:
# 3) Umbrales fijados solo con train: el más alto que todavía alcanza recall >= 0.95 (fuera de pliegue)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

def umbral_train(modelo):
    p = cross_val_predict(clone(modelo), tr.text, ytr, cv=cv, method="predict_proba")[:, 1]
    prec, rec, umbrales = precision_recall_curve(ytr, p)
    return umbrales[rec[:-1] >= META_RECALL].max()

t0 = time.time()
U_LR = umbral_train(lr_f)
U_ST = umbral_train(stacking)
print(f"Umbral LogReg afinado: {U_LR:.4f} | Umbral Stacking: {U_ST:.4f}  ({time.time() - t0:.0f} s)")

Umbral LogReg afinado: 0.3182 | Umbral Stacking: 0.3247  (104 s)


In [4]:
# 4) Evaluación en Kaspersky (una sola corrida)
kas = pd.read_csv("oraciones_kaspersky.csv")
kas = kas[kas["tipo"] == "oracion"].reset_index(drop=True)          # 96 oraciones
et = pd.read_csv("anotacion_referencia.csv")                        # id, label_ref, ttps_gt
k = kas.merge(et[["id", "label_ref"]], on="id", how="inner")
assert len(k) == 96, len(k)
yk = k["label_ref"].to_numpy()                                      # 55 ataques, 41 contexto

def evaluar_kaspersky(nombre, modelo, umbral):
    m = clone(modelo).fit(tr.text, ytr)                             # entrenado solo con train
    pred = (m.predict_proba(k["texto"])[:, 1] >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(yk, pred).ravel()
    return {"modelo": nombre, "umbral": round(float(umbral), 4),
            "recall": recall_score(yk, pred), "precisión": precision_score(yk, pred),
            "F1": f1_score(yk, pred), "F2": fbeta_score(yk, pred, beta=2),
            "TP": tp, "FN": fn, "FP": fp, "TN": tn}

res_kas = pd.DataFrame([evaluar_kaspersky("Baseline Av3 (LogReg)", baseline, 0.5),
                        evaluar_kaspersky("LogReg afinado + umbral de train", lr_f, U_LR),
                        evaluar_kaspersky("Stacking (LogReg + CNB calibrado)", stacking, U_ST)]).round(3)
print(res_kas.to_string(index=False))

                           modelo  umbral  recall  precisión    F1    F2  TP  FN  FP  TN
            Baseline Av3 (LogReg)   0.500   0.364      0.870 0.513 0.412  20  35   3  38
 LogReg afinado + umbral de train   0.318   0.618      0.791 0.694 0.646  34  21   9  32
Stacking (LogReg + CNB calibrado)   0.325   0.636      0.854 0.729 0.670  35  20   6  35


In [5]:
# 5) Control: deben reproducirse los valores reportados en el artículo (Tabla 8)
controles = {"Umbral LogReg ≈ 0.318": abs(U_LR - 0.318) < 0.001,
             "Umbral Stacking ≈ 0.325": abs(U_ST - 0.325) < 0.001,
             "Recall baseline = 0.364": abs(res_kas.loc[0, "recall"] - 0.364) < 0.001,
             "Recall LogReg = 0.618": abs(res_kas.loc[1, "recall"] - 0.618) < 0.001,
             "Recall Stacking = 0.636": abs(res_kas.loc[2, "recall"] - 0.636) < 0.001}
for nombre, ok in controles.items():
    print(f"{nombre}: {'OK' if ok else 'REVISAR'}")

Umbral LogReg ≈ 0.318: OK
Umbral Stacking ≈ 0.325: OK
Recall baseline = 0.364: OK
Recall LogReg = 0.618: OK
Recall Stacking = 0.636: OK


## Valores de referencia (Tabla 8 del artículo)

| Modelo | Recall | Precisión | F1 | F2 | TP | FN | FP | TN |
|---|---|---|---|---|---|---|---|---|
| Baseline | 0.364 | 0.870 | 0.513 | 0.412 | 20 | 35 | 3 | 38 |
| LogReg afinado, umbral 0.318 | 0.618 | 0.791 | 0.694 | 0.646 | 34 | 21 | 9 | 32 |
| Stacking, umbral 0.325 | 0.636 | 0.854 | 0.729 | 0.670 | 35 | 20 | 6 | 35 |